# Simulation: adding a state-level shock

New notebook. The LP picks a portfolio; this asks whether that portfolio holds up when the economy turns.

## What was there before

The scaffold used a single factor. Every simulated year draws one number for the whole country, and that number pushes every loan in the book the same direction:

```
score = √0.15 x national + √0.85 x individual
```

A loan defaults if its score falls below a threshold set from its predicted PD. The 0.15 is Basel's asset correlation for residential mortgages, not a number we chose.

**The problem with that.** One national factor hits every loan identically regardless of where the house is. So spreading a portfolio across states protects against nothing, because there is no state-level risk in the model to protect against. When the scaffold found the state cap bought no measurable downside protection, that result was baked in by the structure.

## What we are adding

A second shared factor, one draw per state per year:

```
score = √ρ_nat x national + √ρ_state x state[i] + √(1 - ρ_nat - ρ_state) x individual
```

Two loans in the same state now share both the national and the state factor. Two loans in different states share only the national one. That is the mechanism a concentration cap is supposed to defend against, and the simulation can finally see it.

**Total correlation stays at 0.15.** We are splitting Basel's number, not adding to it. National share defaults to 79%, which puts national at 0.119 and state at 0.032. That 79% is a placeholder pending the EDA track's check against FHFA house price data, so it is a variable set in one place.

**States are treated as independent**, which is deliberately the strongest case for diversification. Real state housing markets move together, so assuming independence overstates how much spreading helps. If the state cap still buys nothing under an assumption that favors it, the finding is much harder to argue with.

## What this cell checks

Synthetic loans only. No real portfolios until the mechanism is proven.

1. Two loans in the same state should correlate at `ρ_nat + ρ_state`
2. Two loans in different states should correlate at `ρ_nat` alone
3. The simulated default rate should land on the input PD
4. **Setting the national share to 1.0 should reproduce the single-factor model exactly**

Item 4 is the one that matters. If the new code does not reduce to the old code, it is not a generalization of it and something is wrong.

In [2]:
import numpy as np
import polars as pl
import time
from scipy.stats import norm

SEED = 591

# --- the one number to change when the EDA track reports back ---
RHO_TOTAL = 0.15        # Basel asset correlation, residential mortgage
NAT_SHARE = 0.79        # national share of housing risk, placeholder

def factor_split(rho_total=RHO_TOTAL, nat_share=NAT_SHARE):
    """Split total correlation into national, state, and individual pieces."""
    rho_nat = rho_total * nat_share
    rho_state = rho_total * (1.0 - nat_share)
    rho_idio = 1.0 - rho_nat - rho_state
    return rho_nat, rho_state, rho_idio

def simulate_defaults(pd_vals, state_idx, n_states, n_years,
                      rho_total=RHO_TOTAL, nat_share=NAT_SHARE, seed=SEED):
    """
    Vasicek draw with a national factor and a per-state factor.

    pd_vals    : predicted default probability, one per loan
    state_idx  : integer state code, one per loan
    returns    : boolean array, shape (n_years, n_loans)
    """
    rho_nat, rho_state, rho_idio = factor_split(rho_total, nat_share)
    rng = np.random.default_rng(seed)

    threshold = norm.ppf(pd_vals)          # a loan defaults below this
    z_nat = rng.standard_normal((n_years, 1))
    z_state = rng.standard_normal((n_years, n_states))
    eps = rng.standard_normal((n_years, len(pd_vals)))

    score = (
        np.sqrt(rho_nat) * z_nat
        + np.sqrt(rho_state) * z_state[:, state_idx]
        + np.sqrt(rho_idio) * eps
    )
    return score < threshold

def pair_corr_batch(defaults, idx_a, idx_b):
    """Correlation for many loan pairs at once, computed straight from means."""
    a = defaults[:, idx_a].astype(np.float64)
    b = defaults[:, idx_b].astype(np.float64)
    ma, mb = a.mean(axis=0), b.mean(axis=0)
    cov = (a * b).mean(axis=0) - ma * mb
    sa, sb = a.std(axis=0), b.std(axis=0)
    ok = (sa > 0) & (sb > 0)
    out = np.full(len(idx_a), np.nan)
    out[ok] = cov[ok] / (sa[ok] * sb[ok])
    return out

# ============================================================
# validation on synthetic loans
# ============================================================
N_LOANS, N_STATES, N_YEARS, TRUE_PD, N_PAIRS = 6_000, 6, 20_000, 0.05, 2_000

state_idx = np.repeat(np.arange(N_STATES), N_LOANS // N_STATES)
pd_vals = np.full(N_LOANS, TRUE_PD)

rho_nat, rho_state, rho_idio = factor_split()
print("--- factor split ---")
print(f"  total correlation (Basel) : {RHO_TOTAL:.3f}")
print(f"  national share            : {NAT_SHARE:.0%}")
print(f"  rho_national              : {rho_nat:.4f}")
print(f"  rho_state                 : {rho_state:.4f}")
print(f"  rho_individual            : {rho_idio:.4f}")
print(f"  sums to                   : {rho_nat + rho_state + rho_idio:.4f}")

t = time.time()
d = simulate_defaults(pd_vals, state_idx, N_STATES, N_YEARS)
print(f"\nsimulated {N_YEARS:,} years x {N_LOANS:,} loans in {time.time()-t:.1f}s")

# --- build the pair samples once, reused for the control below ---
rng_pairs = np.random.default_rng(SEED)
by_state = [np.flatnonzero(state_idx == s) for s in range(N_STATES)]

pick = rng_pairs.integers(0, N_STATES, N_PAIRS)
same_a = np.array([rng_pairs.choice(by_state[s]) for s in pick])
same_b = np.array([rng_pairs.choice(by_state[s]) for s in pick])
keep = same_a != same_b
same_a, same_b = same_a[keep], same_b[keep]

s1 = rng_pairs.integers(0, N_STATES, N_PAIRS)
s2 = (s1 + rng_pairs.integers(1, N_STATES, N_PAIRS)) % N_STATES
diff_a = np.array([rng_pairs.choice(by_state[s]) for s in s1])
diff_b = np.array([rng_pairs.choice(by_state[s]) for s in s2])

# --- checks 1 and 2: same-state pairs should correlate more ---
c_same = pair_corr_batch(d, same_a, same_b)
c_diff = pair_corr_batch(d, diff_a, diff_b)

print("\n--- realized default correlation between loan pairs ---")
print(f"{'group':18} {'pairs':>7} {'mean':>9} {'std':>9} {'p5':>9} {'p95':>9}")
print("-" * 64)
for label, c in [("same state", c_same), ("different state", c_diff)]:
    print(f"{label:18} {len(c):>7,} {np.nanmean(c):>9.4f} {np.nanstd(c):>9.4f} "
          f"{np.nanpercentile(c, 5):>9.4f} {np.nanpercentile(c, 95):>9.4f}")
print("-" * 64)

gap = np.nanmean(c_same) - np.nanmean(c_diff)
sep = gap / np.sqrt(np.nanvar(c_same)/len(c_same) + np.nanvar(c_diff)/len(c_diff))
print(f"gap (same minus different) : {gap:+.4f}")
print(f"gap in standard errors     : {sep:.1f}")
print("\n  These are DEFAULT correlations and sit well below the ASSET")
print("  correlations in the factor split. The relationship compresses hard")
print("  at low PD. What matters is the gap, not the level.")

# --- check 3: does the mean land on the input PD ---
print(f"\n--- default rate ---")
print(f"  input PD    : {TRUE_PD:.4f}")
print(f"  simulated   : {d.mean():.4f}")
print(f"  worst year  : {d.mean(axis=1).max():.4f}")
print(f"  best year   : {d.mean(axis=1).min():.4f}")

# --- check 4: national share of 1.0 must reproduce the single-factor model ---
def single_factor(pd_vals, n_years, n_states, rho=RHO_TOTAL, seed=SEED):
    """The scaffold's original model. Burns the state draws so the
    individual draws line up with the new code's random stream."""
    rng = np.random.default_rng(seed)
    z = rng.standard_normal((n_years, 1))
    rng.standard_normal((n_years, n_states))
    e = rng.standard_normal((n_years, len(pd_vals)))
    return np.sqrt(rho) * z + np.sqrt(1 - rho) * e < norm.ppf(pd_vals)

d_new = simulate_defaults(pd_vals, state_idx, N_STATES, 2_000, nat_share=1.0)
d_old = single_factor(pd_vals, 2_000, N_STATES)

print("\n--- does national share = 1.0 reproduce the single-factor model ---")
print(f"  identical on every draw : {np.array_equal(d_new, d_old)}")
print(f"  new model rate          : {d_new.mean():.6f}")
print(f"  old model rate          : {d_old.mean():.6f}")

# --- control: with the state factor off, the two pair groups should collapse ---
d_nat_only = simulate_defaults(pd_vals, state_idx, N_STATES, N_YEARS, nat_share=1.0)
c_same_0 = pair_corr_batch(d_nat_only, same_a, same_b)
c_diff_0 = pair_corr_batch(d_nat_only, diff_a, diff_b)

print("\n--- control: national factor only, no state shock ---")
print(f"  same state      : {np.nanmean(c_same_0):.4f}")
print(f"  different state : {np.nanmean(c_diff_0):.4f}")
print(f"  gap             : {np.nanmean(c_same_0) - np.nanmean(c_diff_0):+.4f}")
print("\n  With the state factor off, same-state and different-state pairs")
print("  should be indistinguishable. That is the scaffold's model, and it is")
print("  why a concentration cap could not have helped there.")

--- factor split ---
  total correlation (Basel) : 0.150
  national share            : 79%
  rho_national              : 0.1185
  rho_state                 : 0.0315
  rho_individual            : 0.8500
  sums to                   : 1.0000

simulated 20,000 years x 6,000 loans in 1.8s

--- realized default correlation between loan pairs ---
group                pairs      mean       std        p5       p95
----------------------------------------------------------------
same state           1,997    0.0415    0.0091    0.0261    0.0567
different state      2,000    0.0315    0.0086    0.0175    0.0457
----------------------------------------------------------------
gap (same minus different) : +0.0100
gap in standard errors     : 35.7

  These are DEFAULT correlations and sit well below the ASSET
  correlations in the factor split. The relationship compresses hard
  at low PD. What matters is the gap, not the level.

--- default rate ---
  input PD    : 0.0500
  simulated   : 0.0505
  w

## Finding: the simulation can now see geography

### What changed

The scaffold used the Vasicek single-factor model. Every loan's score is one systematic factor shared across the whole book, plus one idiosyncratic factor per loan:

```
score = √0.15 x systematic + √0.85 x idiosyncratic
```

A loan defaults when its score falls below the threshold set from its PD. The systematic factor is what makes bad years bad: when it lands low, defaults cluster.

**The problem is that the systematic factor was purely national.** Every loan got the same draw regardless of location. That makes the state cap useless by construction, because there is no geographic component to diversify away.

We split the systematic factor in two:

```
score = √0.1185 x national + √0.0315 x state + √0.85 x idiosyncratic
```

Total asset correlation stays at Basel's 0.15. We split it 79/21, we did not add to it.

### What the check shows

| Loan pair | Default correlation |
|---|---|
| Same state | 0.0415 |
| Different states | 0.0315 |

Same-state pairs correlate 32% higher, at 35.7 standard errors. Real, not sampling noise.

Note these are default correlations, not asset correlations. They sit well below the 0.15 input because the mapping from asset to default correlation compresses hard at low PD. The gap between the two groups is the result, not the level.

**Control run, state factor off:**

| Loan pair | Default correlation |
|---|---|
| Same state | 0.0415 |
| Different states | 0.0414 |

They collapse. With no state factor, geography carries no information at all.

**That is the explanation for the scaffold's finding.** The state cap provided no measurable downside protection because the model contained no state-level systematic risk. Not a weakness of the cap, a limit of the simulation.

### Two more checks

- **Setting the national share to 1.0 reproduces the single-factor model exactly**, identical on every draw. The new code is a strict generalization of the old, not a rewrite that changed something by accident.
- **The mean lands on the input PD.** 0.0505 simulated against 0.0500 input across 20,000 draws. Worst draw was 41.85%, so the tail is present.

### One assumption worth naming

State factors are drawn independently. Real state housing markets are positively correlated with each other, so independence overstates how much geographic diversification helps. 

That is deliberate. It is the strongest case for the state cap. If the cap still buys nothing under an assumption tilted in its favor, the finding is much harder to argue with.

## Base run: six portfolios, national and state shocks

**What happens.** Simulate 10,000 seven-year outcomes for each portfolio. Every year draws one national factor, one factor per state, and one idiosyncratic factor per loan. A loan defaults when its combined score falls below the threshold set from its PD.

**Every portfolio sees the same 10,000 years.** Same draws, same shocks. Any difference between them comes from which loans they hold, not from luck.

**Return in a simulated year:**

```
return = interest on loans that paid  -  loss on loans that defaulted
```

Loss is `UPB x LGD`, LGD flat at 30% following Sirignano et al. Every recovery field in the Fannie data is null across all 2 million loans, so a loan-level LGD model was not possible.

**Chunked over years** because a 10,000 by 410,000 boolean array is about 4GB. 500 years at a time. Same results, less memory.

**The columns**

- **avg**: average return across all 10,000 years
- **std**: how much the outcomes spread out
- **bad year**: the return you beat 95% of the time. Only 500 of 10,000 years came in worse.
- **worst 5%**: the average across those 500 worst years. This is expected shortfall, the Basel Committee's post-2008 replacement for Value-at-Risk. "Bad year" tells you where the bad tail starts, "worst 5%" tells you how bad it gets once you are in it.
- **worst**: the single worst of the 10,000
- **yrs w/ loss**: how often the portfolio lost money

In [5]:
import polars as pl
import numpy as np
import time
from pathlib import Path
from scipy.stats import norm

PROC = Path("../data/processed")
port = pl.read_parquet(PROC / "prod_portfolios.parquet")

# label, score, rule, column suffix
PORTFOLIOS = [
    ("rule-based", "risk-sort",     "rule_risk_sort"),
    ("rule-based", "greedy-return", "rule_greedy_return"),
    ("rule-based", "LP",            "rule_lp"),
    ("CatBoost",   "risk-sort",     "cat_risk_sort"),
    ("CatBoost",   "greedy-return", "cat_greedy_return"),
    ("CatBoost",   "LP",            "cat_lp"),
]
KEYS = [k for _, _, k in PORTFOLIOS]

N_YEARS = 10_000
CHUNK = 500
LGD = 0.30

pd_loan = port["pd"].to_numpy()
threshold = norm.ppf(pd_loan)
interest = port["interest_income_7yr"].to_numpy()
upb_loan = port["ORIG_UPB"].to_numpy()

state_codes = port["STATE"].to_numpy()
state_names = sorted(set(state_codes))
state_idx = np.array([state_names.index(s) for s in state_codes])
n_states = len(state_names)

weights = {k: port[f"x_{k}"].to_numpy() for k in KEYS}
funded_dollars = {k: float((w * upb_loan).sum()) for k, w in weights.items()}

print(f"loans   : {len(port):,}")
print(f"states  : {n_states}")
print(f"years   : {N_YEARS:,}   in chunks of {CHUNK}")
print(f"budget  : ${funded_dollars['cat_lp']/1e9:,.3f}B\n")

def run_simulation(rho_total=RHO_TOTAL, nat_share=NAT_SHARE,
                   lgd=LGD, n_years=N_YEARS, seed=SEED):
    """Simulate every portfolio over the same set of years. Returns dollars."""
    rho_nat, rho_state, rho_idio = factor_split(rho_total, nat_share)
    sq_nat, sq_state, sq_idio = np.sqrt([rho_nat, rho_state, rho_idio])
    rng = np.random.default_rng(seed)

    loss_loan = upb_loan * lgd
    pay = {k: weights[k] * interest for k in KEYS}
    hit = {k: weights[k] * loss_loan for k in KEYS}
    out = {k: np.empty(n_years) for k in KEYS}

    for start in range(0, n_years, CHUNK):
        j = min(CHUNK, n_years - start)
        z_nat = rng.standard_normal((j, 1))
        z_state = rng.standard_normal((j, n_states))
        eps = rng.standard_normal((j, len(port)))

        score = sq_nat * z_nat + sq_state * z_state[:, state_idx] + sq_idio * eps
        defaulted = score < threshold

        for k in KEYS:
            out[k][start:start + j] = pay[k].sum() - defaulted @ (pay[k] + hit[k])
        del score, defaulted

    return out

t = time.time()
returns = run_simulation()
print(f"simulated {N_YEARS:,} years x 6 portfolios in {(time.time()-t)/60:.1f} min\n")

def metrics(r, funded):
    pct = r / funded * 100
    bad = np.percentile(pct, 5)
    return {
        "avg": pct.mean(),
        "std": pct.std(),
        "bad_year": bad,
        "worst_5pct": pct[pct <= bad].mean(),
        "worst": pct.min(),
        "loss_years": int((pct < 0).sum()),
    }

stats = {k: metrics(returns[k], funded_dollars[k]) for k in KEYS}

print("Return on dollars funded, over 7 years.")
print(f"\n  {'score':11} {'rule':15} {'avg':>8} {'std':>7} {'bad year':>10} "
      f"{'worst 5%':>10} {'worst':>9} {'yrs w/ loss':>12}")
print("  " + "-" * 88)
for score, rule, k in PORTFOLIOS:
    s = stats[k]
    print(f"  {score:11} {rule:15} {s['avg']:>7.2f}% {s['std']:>7.2f} "
          f"{s['bad_year']:>9.2f}% {s['worst_5pct']:>9.2f}% "
          f"{s['worst']:>8.2f}% {s['loss_years']:>11,}")
print("  " + "-" * 88)
print("  bad year = beaten 95% of the time    worst 5% = average of the worst 500 years")

# --- all three rules, side by side, one metric at a time ---
METRICS = [("avg", "average"), ("std", "spread"), ("bad_year", "bad year"),
           ("worst_5pct", "worst 5%"), ("worst", "worst year")]

print("\n\nAll three rules, same score:")
for score, prefix in [("rule-based", "rule"), ("CatBoost", "cat")]:
    print(f"\n  {score}")
    print(f"    {'':14} {'risk-sort':>11} {'greedy-return':>15} {'LP':>11}")
    print("    " + "-" * 53)
    for key, label in METRICS:
        r = stats[f"{prefix}_risk_sort"][key]
        g = stats[f"{prefix}_greedy_return"][key]
        l = stats[f"{prefix}_lp"][key]
        unit = "" if key == "std" else "%"
        print(f"    {label:14} {r:>10.2f}{unit} {g:>14.2f}{unit} {l:>10.2f}{unit}")

# --- what the better score buys, one rule at a time ---
print("\n\nCatBoost minus rule-based, same rule:")
print(f"  {'':14} {'risk-sort':>11} {'greedy-return':>15} {'LP':>11}")
print("  " + "-" * 53)
for key, label in METRICS:
    row = []
    for suffix in ["risk_sort", "greedy_return", "lp"]:
        row.append(stats[f"cat_{suffix}"][key] - stats[f"rule_{suffix}"][key])
    print(f"  {label:14} {row[0]:>+10.2f}  {row[1]:>+14.2f}  {row[2]:>+10.2f}")

# --- what the constraints cost, one score at a time ---
print("\n\nLP minus greedy-return, same score:")
print(f"  {'':14} {'rule-based':>12} {'CatBoost':>12}")
print("  " + "-" * 40)
for key, label in METRICS:
    a = stats["rule_lp"][key] - stats["rule_greedy_return"][key]
    b = stats["cat_lp"][key] - stats["cat_greedy_return"][key]
    print(f"  {label:14} {a:>+11.2f}  {b:>+11.2f}")

loans   : 409,857
states  : 54
years   : 10,000   in chunks of 500
budget  : $9.376B

simulated 10,000 years x 6 portfolios in 2.1 min

Return on dollars funded, over 7 years.

  score       rule                 avg     std   bad year   worst 5%     worst  yrs w/ loss
  ----------------------------------------------------------------------------------------
  rule-based  risk-sort         24.22%    0.50     23.25%     22.64%    19.36%           0
  rule-based  greedy-return     29.43%    1.83     25.78%     24.18%    16.37%           0
  rule-based  LP                29.16%    1.82     25.58%     23.96%    15.40%           0
  CatBoost    risk-sort         23.32%    0.22     22.89%     22.59%    20.51%           0
  CatBoost    greedy-return     30.14%    1.31     27.50%     26.22%    19.60%           0
  CatBoost    LP                29.83%    1.29     27.24%     25.98%    18.82%           0
  ----------------------------------------------------------------------------------------
  b

## Diagnostic: is the state factor actually reaching these portfolios?

The base run says the concentration cap does not help. Before that goes in the report, confirm the mechanism is firing on real data.

**Why this needs checking.** The synthetic validation used 6 states holding 1,000 loans each. Here there are 54 states, California holds 56,162 loans and the smallest hold a few hundred. The math is the same but the shape of the data is not.

**Three checks**

- **Do state default rates spread out?** In the worst simulated years, individual states should diverge from the national average. If every state moves in lockstep, the state factor is not doing anything.
- **Does California concentration show up as volatility?** Greedy holds 28.2% of its budget in California, the LP holds 6%. When the California draw goes bad, greedy should take a hit the LP does not. That is the exact mechanism the cap is supposed to defend against.
- **What happens in California's worst years specifically?** Isolate the years where the California factor landed worst and compare the two portfolios there.

In [7]:
# --- rerun a smaller simulation, keeping the state draws this time ---
DIAG_YEARS = 2_000

def run_with_state_draws(rho_total=RHO_TOTAL, nat_share=NAT_SHARE,
                         lgd=LGD, n_years=DIAG_YEARS, seed=SEED):
    """Same simulation, but hands back the state factors and per-state
    default rates so we can see the mechanism instead of just the totals."""
    rho_nat, rho_state, rho_idio = factor_split(rho_total, nat_share)
    sq_nat, sq_state, sq_idio = np.sqrt([rho_nat, rho_state, rho_idio])
    rng = np.random.default_rng(seed)

    loss_loan = upb_loan * lgd
    pay = {k: weights[k] * interest for k in KEYS}
    hit = {k: weights[k] * loss_loan for k in KEYS}

    out = {k: np.empty(n_years) for k in KEYS}
    z_state_all = np.empty((n_years, n_states))
    z_nat_all = np.empty(n_years)
    state_rates = np.empty((n_years, n_states))

    # which loans sit in which state, and their dollar weight
    state_mask = [state_idx == s for s in range(n_states)]

    for start in range(0, n_years, CHUNK):
        j = min(CHUNK, n_years - start)
        z_nat = rng.standard_normal((j, 1))
        z_state = rng.standard_normal((j, n_states))
        eps = rng.standard_normal((j, len(port)))

        score = sq_nat * z_nat + sq_state * z_state[:, state_idx] + sq_idio * eps
        defaulted = score < threshold

        z_nat_all[start:start + j] = z_nat[:, 0]
        z_state_all[start:start + j] = z_state
        for s in range(n_states):
            state_rates[start:start + j, s] = defaulted[:, state_mask[s]].mean(axis=1)

        for k in KEYS:
            out[k][start:start + j] = pay[k].sum() - defaulted @ (pay[k] + hit[k])
        del score, defaulted

    return out, z_nat_all, z_state_all, state_rates

t = time.time()
dret, z_nat_all, z_state_all, state_rates = run_with_state_draws()
print(f"diagnostic run: {DIAG_YEARS:,} years in {time.time()-t:.1f}s\n")

ca = state_names.index("CA")

# ---- 1. do state default rates spread out ----
national_rate = state_rates.mean(axis=1)
worst_idx = np.argsort(national_rate)[-20:]      # 20 worst national years
best_idx = np.argsort(national_rate)[:20]

print("--- 1. spread of state default rates within a single year ---")
print(f"{'':22} {'mean rate':>11} {'spread across states':>22} {'min':>8} {'max':>8}")
print("-" * 74)
for label, idx in [("20 best years", best_idx), ("all years", np.arange(DIAG_YEARS)),
                   ("20 worst years", worst_idx)]:
    r = state_rates[idx]
    print(f"{label:22} {r.mean()*100:>10.3f}% {r.std(axis=1).mean()*100:>21.3f}% "
          f"{r.min()*100:>7.3f}% {r.max()*100:>7.3f}%")
print("-" * 74)
print("  spread > 0 means states diverge from each other, which is the state factor working")

# ---- 2. does the California draw move the portfolios differently ----
cat_greedy_pct = dret["cat_greedy_return"] / funded_dollars["cat_greedy_return"] * 100
cat_lp_pct = dret["cat_lp"] / funded_dollars["cat_lp"] * 100
gap = cat_greedy_pct - cat_lp_pct

print(f"\n--- 2. does the CA factor drive the gap between greedy and LP ---")
print(f"  greedy holds 28.2% of budget in CA, the LP holds 6.0%")
print(f"\n  correlation, CA factor vs (greedy minus LP)  : "
      f"{np.corrcoef(z_state_all[:, ca], gap)[0,1]:>+.4f}")
print(f"  correlation, national factor vs (greedy - LP) : "
      f"{np.corrcoef(z_nat_all, gap)[0,1]:>+.4f}")
print("\n  A positive CA correlation means greedy does better when CA does well")
print("  and worse when CA does badly. That is the concentration risk the cap targets.")

# ---- 3. what happens in California's worst years ----
ca_worst = np.argsort(z_state_all[:, ca])[:100]        # 100 worst CA draws
ca_best = np.argsort(z_state_all[:, ca])[-100:]

print(f"\n--- 3. CatBoost greedy vs LP, split by how CA did ---")
print(f"{'':24} {'greedy':>10} {'LP':>10} {'greedy - LP':>13}")
print("-" * 60)
for label, idx in [("100 best CA years", ca_best),
                   ("all years", np.arange(DIAG_YEARS)),
                   ("100 worst CA years", ca_worst)]:
    g, l = cat_greedy_pct[idx].mean(), cat_lp_pct[idx].mean()
    print(f"{label:24} {g:>9.2f}% {l:>9.2f}% {g-l:>+12.2f}")
print("-" * 60)

print(f"\n  CA default rate in its 100 worst years : "
      f"{state_rates[ca_worst, ca].mean()*100:.3f}%")
print(f"  CA default rate in its 100 best years  : "
      f"{state_rates[ca_best, ca].mean()*100:.3f}%")
print(f"  CA default rate overall                : "
      f"{state_rates[:, ca].mean()*100:.3f}%")

diagnostic run: 2,000 years in 26.7s

--- 1. spread of state default rates within a single year ---
                         mean rate   spread across states      min      max
--------------------------------------------------------------------------
20 best years               0.292%                 0.356%   0.000%   4.545%
all years                   3.372%                 2.179%   0.000%  54.545%
20 worst years             14.465%                 5.974%   0.000%  54.545%
--------------------------------------------------------------------------
  spread > 0 means states diverge from each other, which is the state factor working

--- 2. does the CA factor drive the gap between greedy and LP ---
  greedy holds 28.2% of budget in CA, the LP holds 6.0%

  correlation, CA factor vs (greedy minus LP)  : +0.7781
  correlation, national factor vs (greedy - LP) : +0.0370

  A positive CA correlation means greedy does better when CA does well
  and worse when CA does badly. That is the concen

## Sweep the national share

The whole state-shock result rests on one number: 79% of housing risk being national. That figure is sourced, but it is a single estimate. This measures how much the conclusion depends on it.

**What changes.** Only the split between the national and state factors. Total asset correlation stays at Basel's 0.15 in every run, so we are moving risk between the two shared factors, not adding any.

| National share | National | State |
|---|---|---|
| 100% | 0.150 | 0.000 |
| 90% | 0.135 | 0.015 |
| 79% | 0.119 | 0.032 |
| 65% | 0.098 | 0.053 |
| 50% | 0.075 | 0.075 |

**The 100% run reproduces the scaffold**, where there was no state factor at all. So the sweep shows the whole progression from the old model to a heavily regional one in one table.

**What we are looking for.** The concentration cap costs the LP about 0.31 points of average return. The question is how much state-level risk it takes before the protection covers that cost. If the crossover happens somewhere in this range, we can name it.

**2,000 years per run** instead of 10,000. Enough to see the direction and find the crossover. The headline numbers stay at 10,000.

In [8]:
NAT_SHARES = [1.00, 0.90, 0.79, 0.65, 0.50]
SWEEP_YEARS = 2_000

rows = []
t0 = time.time()

for share in NAT_SHARES:
    t = time.time()
    r = run_simulation(nat_share=share, n_years=SWEEP_YEARS)
    rho_n, rho_s, _ = factor_split(nat_share=share)

    for score, rule, k in PORTFOLIOS:
        s = metrics(r[k], funded_dollars[k])
        rows.append({
            "nat_share": share,
            "rho_nat": rho_n,
            "rho_state": rho_s,
            "score": score,
            "rule": rule,
            "key": k,
            **s,
        })
    print(f"  national share {share:.0%}  "
          f"(state rho {rho_s:.4f})  done in {time.time()-t:.1f}s")

print(f"\nswept {len(NAT_SHARES)} settings in {(time.time()-t0)/60:.1f} min\n")
sweep_nat = pl.DataFrame(rows)

# ---- the full table, one block per setting ----
for share in NAT_SHARES:
    block = sweep_nat.filter(pl.col("nat_share") == share)
    rho_s = block["rho_state"][0]
    print(f"national share {share:.0%}   state asset correlation {rho_s:.4f}")
    print(f"  {'score':11} {'rule':15} {'avg':>8} {'std':>7} {'bad year':>10} "
          f"{'worst 5%':>10} {'worst':>9}")
    print("  " + "-" * 74)
    for row in block.iter_rows(named=True):
        print(f"  {row['score']:11} {row['rule']:15} {row['avg']:>7.2f}% "
              f"{row['std']:>7.2f} {row['bad_year']:>9.2f}% "
              f"{row['worst_5pct']:>9.2f}% {row['worst']:>8.2f}%")
    print()

# ---- the question: where does the LP catch greedy ----
print("=" * 78)
print("LP minus greedy-return, CatBoost score")
print("=" * 78)
print(f"  {'nat share':>10} {'state rho':>10} {'average':>10} {'bad year':>10} "
      f"{'worst 5%':>10} {'worst year':>12}")
print("  " + "-" * 66)
for share in NAT_SHARES:
    b = sweep_nat.filter(pl.col("nat_share") == share)
    lp = b.filter(pl.col("key") == "cat_lp").row(0, named=True)
    gr = b.filter(pl.col("key") == "cat_greedy_return").row(0, named=True)
    print(f"  {share:>9.0%} {lp['rho_state']:>10.4f} "
          f"{lp['avg']-gr['avg']:>+9.2f} {lp['bad_year']-gr['bad_year']:>+9.2f} "
          f"{lp['worst_5pct']-gr['worst_5pct']:>+9.2f} "
          f"{lp['worst']-gr['worst']:>+11.2f}")
print("  " + "-" * 66)
print("  positive = the LP wins   negative = greedy wins")

print("\n" + "=" * 78)
print("Same, rule-based score")
print("=" * 78)
print(f"  {'nat share':>10} {'state rho':>10} {'average':>10} {'bad year':>10} "
      f"{'worst 5%':>10} {'worst year':>12}")
print("  " + "-" * 66)
for share in NAT_SHARES:
    b = sweep_nat.filter(pl.col("nat_share") == share)
    lp = b.filter(pl.col("key") == "rule_lp").row(0, named=True)
    gr = b.filter(pl.col("key") == "rule_greedy_return").row(0, named=True)
    print(f"  {share:>9.0%} {lp['rho_state']:>10.4f} "
          f"{lp['avg']-gr['avg']:>+9.2f} {lp['bad_year']-gr['bad_year']:>+9.2f} "
          f"{lp['worst_5pct']-gr['worst_5pct']:>+9.2f} "
          f"{lp['worst']-gr['worst']:>+11.2f}")
print("  " + "-" * 66)

  national share 100%  (state rho 0.0000)  done in 26.1s
  national share 90%  (state rho 0.0150)  done in 25.3s
  national share 79%  (state rho 0.0315)  done in 24.5s
  national share 65%  (state rho 0.0525)  done in 24.9s
  national share 50%  (state rho 0.0750)  done in 25.0s

swept 5 settings in 2.1 min

national share 100%   state asset correlation 0.0000
  score       rule                 avg     std   bad year   worst 5%     worst
  --------------------------------------------------------------------------
  rule-based  risk-sort         24.22%    0.60     23.08%     22.26%    19.18%
  rule-based  greedy-return     29.40%    2.12     25.28%     23.07%    16.15%
  rule-based  LP                29.13%    2.11     25.06%     22.84%    16.01%
  CatBoost    risk-sort         23.32%    0.27     22.83%     22.39%    20.45%
  CatBoost    greedy-return     30.12%    1.53     27.11%     25.34%    19.24%
  CatBoost    LP                29.81%    1.51     26.85%     25.11%    19.13%

natio

## Check the endpoints at full resolution

The sweep says the LP's disadvantage is flat at -0.31 no matter how much state-level risk exists. That is the headline result, so the tail numbers behind it need to be measured rather than estimated from a short run.

**What this does.** Reruns the two endpoint settings, 100% national and 50% national, at 10,000 years instead of 2,000. That is the same resolution as the base run.

**What we are separating.** In the 2,000-year sweep the worst-5% gap wandered from -0.24 to -0.31 to -0.23 with no clear pattern. Either that is real movement or it is sampling noise. At 10,000 years the tail metrics settle down enough to tell the difference.

**Also checking the symmetry explanation.** The diagnostic showed greedy gaining in California's good years and losing in its bad years. If those cancel, that explains why the average gap does not move. This splits each run's years into the best and worst thirds by national outcome and reports the gap in each, so we can see the cancellation directly instead of inferring it.

In [9]:
ENDPOINTS = [1.00, 0.50]
FULL_YEARS = 10_000

endpoint_returns = {}
rows = []

for share in ENDPOINTS:
    t = time.time()
    r = run_simulation(nat_share=share, n_years=FULL_YEARS)
    endpoint_returns[share] = r
    rho_n, rho_s, _ = factor_split(nat_share=share)
    for score, rule, k in PORTFOLIOS:
        rows.append({"nat_share": share, "rho_state": rho_s,
                     "score": score, "rule": rule, "key": k,
                     **metrics(r[k], funded_dollars[k])})
    print(f"  national share {share:.0%}  {FULL_YEARS:,} years  "
          f"in {time.time()-t:.1f}s")

endpoints = pl.DataFrame(rows)

print("\n" + "=" * 78)
print(f"LP minus greedy-return, {FULL_YEARS:,} years")
print("=" * 78)
print(f"  {'score':11} {'nat share':>10} {'state rho':>10} {'average':>10} "
      f"{'bad year':>10} {'worst 5%':>10} {'worst year':>12}")
print("  " + "-" * 76)
for score, prefix in [("rule-based", "rule"), ("CatBoost", "cat")]:
    for share in ENDPOINTS:
        b = endpoints.filter(pl.col("nat_share") == share)
        lp = b.filter(pl.col("key") == f"{prefix}_lp").row(0, named=True)
        gr = b.filter(pl.col("key") == f"{prefix}_greedy_return").row(0, named=True)
        print(f"  {score:11} {share:>9.0%} {lp['rho_state']:>10.4f} "
              f"{lp['avg']-gr['avg']:>+9.2f} "
              f"{lp['bad_year']-gr['bad_year']:>+9.2f} "
              f"{lp['worst_5pct']-gr['worst_5pct']:>+9.2f} "
              f"{lp['worst']-gr['worst']:>+11.2f}")
print("  " + "-" * 76)
print("  positive = the LP wins   negative = greedy wins")

# ---- does the gap cancel out between good and bad years ----
print("\n" + "=" * 78)
print("Where the gap comes from: good years against bad years, CatBoost score")
print("=" * 78)
for share in ENDPOINTS:
    g = endpoint_returns[share]["cat_greedy_return"] / funded_dollars["cat_greedy_return"] * 100
    l = endpoint_returns[share]["cat_lp"] / funded_dollars["cat_lp"] * 100
    gap = l - g

    order = np.argsort(g)                    # sort years by how greedy did
    third = FULL_YEARS // 3
    buckets = [("worst third", order[:third]),
               ("middle third", order[third:2*third]),
               ("best third", order[2*third:])]

    rho_s = factor_split(nat_share=share)[1]
    print(f"\n  national share {share:.0%}   state asset correlation {rho_s:.4f}")
    print(f"    {'':16} {'greedy':>10} {'LP':>10} {'LP - greedy':>13}")
    print("    " + "-" * 52)
    for label, idx in buckets:
        print(f"    {label:16} {g[idx].mean():>9.2f}% {l[idx].mean():>9.2f}% "
              f"{gap[idx].mean():>+12.2f}")
    print(f"    {'all years':16} {g.mean():>9.2f}% {l.mean():>9.2f}% "
          f"{gap.mean():>+12.2f}")

print("\n  If the LP's gap shrinks or flips in the worst third, the cap is")
print("  buying downside protection and paying for it with upside.")

# ---- how much do the two portfolios' outcomes diverge ----
print("\n" + "=" * 78)
print("How closely do the two portfolios move together")
print("=" * 78)
for share in ENDPOINTS:
    g = endpoint_returns[share]["cat_greedy_return"] / funded_dollars["cat_greedy_return"] * 100
    l = endpoint_returns[share]["cat_lp"] / funded_dollars["cat_lp"] * 100
    rho_s = factor_split(nat_share=share)[1]
    print(f"  national share {share:.0%}  (state rho {rho_s:.4f})   "
          f"correlation {np.corrcoef(g, l)[0,1]:.5f}   "
          f"spread of the gap {(l-g).std():.4f}")
print("\n  A correlation near 1.0 means the two portfolios rise and fall together,")
print("  so the cap is not changing what the portfolio is exposed to.")

  national share 100%  10,000 years  in 124.5s
  national share 50%  10,000 years  in 121.9s

LP minus greedy-return, 10,000 years
  score        nat share  state rho    average   bad year   worst 5%   worst year
  ----------------------------------------------------------------------------
  rule-based       100%     0.0000     -0.27     -0.26     -0.24       -0.14
  rule-based        50%     0.0750     -0.27     -0.16     -0.13       -1.15
  CatBoost         100%     0.0000     -0.31     -0.27     -0.25       -0.24
  CatBoost          50%     0.0750     -0.31     -0.18     -0.16       -0.94
  ----------------------------------------------------------------------------
  positive = the LP wins   negative = greedy wins

Where the gap comes from: good years against bad years, CatBoost score

  national share 100%   state asset correlation 0.0000
                         greedy         LP   LP - greedy
    ----------------------------------------------------
    worst third          28.5

## Finding: the concentration cap works, and still does not pay

The state shock is in, the mechanism fires, and the answer does not change. That is a stronger result than the scaffold's, which could only say the cap did nothing.

### The mechanism is real and measurable

The gap between the greedy portfolio and the LP correlates **+0.78 with the California factor** and **+0.04 with the national factor**. Greedy holds 28.2% of its budget in California, the LP holds 6%. The difference between those two portfolios is almost entirely a bet on one state, measured directly rather than assumed.

State default rates spread apart by 2.18 points on average and 5.97 points in the worst national years. California's own default rate swings from 1.51% in its best years to 6.64% in its worst.

### The cap buys downside protection

CatBoost score, LP minus greedy, 10,000 years, split by how greedy did:

| | No state factor | Strong state factor |
|---|---|---|
| Worst third of years | -0.29 | **-0.24** |
| Middle third | -0.31 | -0.33 |
| Best third | -0.32 | **-0.35** |
| All years | -0.31 | -0.31 |

With no state factor the gap is flat across all three thirds, because there is nothing regional for the cap to protect against. Add state risk and it tilts: the LP gives back 0.11 points in bad years and pays 0.03 more in good ones.

The tail metrics move the same way. The worst-5% gap improves from -0.25 to -0.16 as state risk rises.

**The two portfolios genuinely diverge.** Their returns correlate 0.99970 with no state factor and 0.97127 with a strong one, and the spread of the gap between them widens six-fold, from 0.041 to 0.251. The cap changes what the portfolio is exposed to. That is exactly what a concentration limit is supposed to do.

### It still does not cover its cost

| National share | State correlation | Average | Bad year | Worst 5% |
|---|---|---|---|---|
| 100% | 0.0000 | -0.31 | -0.27 | -0.25 |
| 90% | 0.0150 | -0.31 | -0.19 | -0.30 |
| 79% (sourced) | 0.0315 | -0.31 | -0.17 | -0.31 |
| 65% | 0.0525 | -0.31 | -0.21 | -0.29 |
| 50% | 0.0750 | -0.31 | -0.18 | -0.23 |

**The average gap reads -0.31 at every setting.** The protection the cap buys in bad years is paid for, almost exactly, by the upside it gives up in good ones. It recovers about a third of its cost at the worst end and never reaches break-even.

**The conclusion does not depend on the 79% figure.** That number is sourced but it is a single estimate, and the whole state-shock result rested on it. Doubling the state factor's strength does not move the answer. Even at a 50/50 split, well beyond anything the literature supports, greedy wins on every metric.

### What this means

Concentration limits are not free risk reduction. They are a trade, and in this pool the trade is roughly symmetric: give up upside in good years, get back a similar amount in bad ones, and pay a structural cost on top.

That structural cost is where greedy's advantage lives. Greedy funds the highest-return loans in the pool regardless of where they sit, and California happens to hold a lot of them. Forcing the money elsewhere means funding worse loans, and no amount of regional diversification recovers what those loans would have earned.

**One assumption still favors the cap.** State factors are drawn independently, so a bad year in California says nothing about Nevada. Real state housing markets move together, which would make diversification worth less than modeled here, not more. The finding holds under an assumption tilted in the cap's favor.

**What this is not.** This is a 2017 book scored against 2017 outcomes, and California was not in a housing downturn during that window. A regional collapse of the kind that hit Nevada, Arizona, and Florida in 2008 is outside anything this simulation draws, because the state factor is a normal distribution and 2008 was not.

## Sensitivity grid: asset correlation crossed with LGD

The last sweep moved risk between the national and state factors. This moves the total amount of risk, and the severity of each loss.

**Asset correlation: 0.00, 0.15, 0.30.** How much loans move together. At 0.00 every default is independent and bad years barely exist. 0.15 is Basel's supervisory value for residential mortgages. 0.30 is double it, well past anything a regulator would require.

**LGD: 30% and 50%.** Loss given default, the share of the loan balance that is actually lost. Both figures come from Sirignano et al. Every recovery field in the Fannie data is null across all 2 million loans, so a loan-level LGD model was not possible and a flat rate is the honest alternative.

**The national/state split stays at 79/21** in every run, so the state factor scales with the total rather than disappearing.

**What we are testing.** Whether the ranking of the six portfolios survives when the risk assumptions move. If greedy beats the LP at every setting, that is not an artifact of the specific numbers we chose. If the ranking flips somewhere, we need to say where and why.

In [11]:
CORRELATIONS = [0.00, 0.15, 0.30]
LGDS = [0.30, 0.50]
GRID_YEARS = 2_000

rows = []
t0 = time.time()

for rho in CORRELATIONS:
    for lgd in LGDS:
        t = time.time()
        r = run_simulation(rho_total=rho, lgd=lgd, n_years=GRID_YEARS)
        for score, rule, k in PORTFOLIOS:
            rows.append({"rho": rho, "lgd": lgd, "score": score,
                         "rule": rule, "key": k,
                         **metrics(r[k], funded_dollars[k])})
        print(f"  correlation {rho:.2f}  LGD {lgd:.0%}   done in {time.time()-t:.1f}s")

print(f"\nall {len(CORRELATIONS)*len(LGDS)} runs in {(time.time()-t0)/60:.1f} min\n")
grid = pl.DataFrame(rows)

for rho in CORRELATIONS:
    for lgd in LGDS:
        block = grid.filter((pl.col("rho") == rho) & (pl.col("lgd") == lgd))
        print(f"asset correlation {rho:.2f}   LGD {lgd:.0%}")
        print(f"  {'score':11} {'rule':15} {'avg':>8} {'std':>7} "
              f"{'bad year':>10} {'worst 5%':>10} {'worst':>9} {'yrs w/ loss':>12}")
        print("  " + "-" * 87)
        for row in block.iter_rows(named=True):
            print(f"  {row['score']:11} {row['rule']:15} {row['avg']:>7.2f}% "
                  f"{row['std']:>7.2f} {row['bad_year']:>9.2f}% "
                  f"{row['worst_5pct']:>9.2f}% {row['worst']:>8.2f}% "
                  f"{row['loss_years']:>11,}")
        print()

# ---- did the ranking hold anywhere ----
print("=" * 80)
print("Does the ranking change anywhere in the grid?")
print("=" * 80)
print(f"  {'correlation':>12} {'LGD':>6} {'metric':>12} "
      f"{'CatBoost LP - greedy':>22} {'CatBoost - rule-based, LP':>27}")
print("  " + "-" * 82)
for rho in CORRELATIONS:
    for lgd in LGDS:
        b = grid.filter((pl.col("rho") == rho) & (pl.col("lgd") == lgd))
        get = lambda k: b.filter(pl.col("key") == k).row(0, named=True)
        for key, label in [("avg", "average"), ("worst_5pct", "worst 5%")]:
            lp_vs_greedy = get("cat_lp")[key] - get("cat_greedy_return")[key]
            cat_vs_rule = get("cat_lp")[key] - get("rule_lp")[key]
            print(f"  {rho:>12.2f} {lgd:>5.0%} {label:>12} "
                  f"{lp_vs_greedy:>+21.2f} {cat_vs_rule:>+26.2f}")
print("  " + "-" * 82)
print("  negative in column 4 = greedy beats the LP")
print("  positive in column 5 = CatBoost beats the rule-based score")

print(f"\nlosing years across all {len(rows)} portfolio-runs: "
      f"{int(grid['loss_years'].sum())}")

# ---- save everything ----
import json
from datetime import datetime, timezone

returns_df = pl.DataFrame({k: returns[k] / funded_dollars[k] * 100 for k in KEYS})
returns_df.write_parquet(PROC / "prod_sim_returns.parquet")

sim_summary = {
    "saved_at": datetime.now(timezone.utc).isoformat(),
    "model": "Vasicek, national factor + per-state factor + idiosyncratic",
    "settings": {
        "rho_total": RHO_TOTAL,
        "rho_total_source": "Basel II/III supervisory asset correlation, "
                            "residential mortgage exposures",
        "nat_share": NAT_SHARE,
        "nat_share_source": "national factor explains ~79% of state-level "
                            "housing price variation; placeholder pending "
                            "EDA check against FHFA data",
        "lgd": LGD,
        "lgd_source": "Sirignano, Tsoukalas, Giesecke (2016); all recovery "
                      "fields null in the Fannie data",
        "n_years": N_YEARS,
        "states_independent": True,
        "independence_note": "deliberately the strongest case for "
                             "diversification; real state markets are "
                             "positively correlated",
    },
    "base_run": {k: metrics(returns[k], funded_dollars[k]) for k in KEYS},
    "national_share_sweep": sweep_nat.to_dicts(),
    "endpoint_runs_10k": endpoints.to_dicts(),
    "sensitivity_grid": grid.to_dicts(),
}
(PROC / "prod_sim_summary.json").write_text(json.dumps(sim_summary, indent=2))

print(f"\nsaved prod_sim_returns.parquet   {returns_df.shape}")
print("saved prod_sim_summary.json")

  correlation 0.00  LGD 30%   done in 26.2s
  correlation 0.00  LGD 50%   done in 24.2s
  correlation 0.15  LGD 30%   done in 24.2s
  correlation 0.15  LGD 50%   done in 24.1s
  correlation 0.30  LGD 30%   done in 24.4s
  correlation 0.30  LGD 50%   done in 24.5s

all 6 runs in 2.5 min

asset correlation 0.00   LGD 30%
  score       rule                 avg     std   bad year   worst 5%     worst  yrs w/ loss
  ---------------------------------------------------------------------------------------
  rule-based  risk-sort         24.22%    0.03     24.17%     24.16%    24.11%           0
  rule-based  greedy-return     29.43%    0.07     29.32%     29.29%    29.13%           0
  rule-based  LP                29.16%    0.07     29.05%     29.02%    28.87%           0
  CatBoost    risk-sort         23.32%    0.02     23.29%     23.29%    23.26%           0
  CatBoost    greedy-return     30.14%    0.05     30.05%     30.03%    29.95%           0
  CatBoost    LP                29.83%    

## Finding: the ranking holds under every stress we tested

Six runs. Asset correlation at 0.00, 0.15, and 0.30, crossed with LGD at 30% and 50%. That range covers from no correlation at all up to double Basel's value, with a two-thirds jump in how much each default costs.

### Nothing changed the order

**Greedy beats the LP in all six runs**, by 0.29 to 0.35 points. The gap barely moves across the whole grid. The concentration cap costs the same whether defaults are independent or heavily clustered.

**CatBoost beats the rule-based score in all six runs**, on both average return and in the bad years.

### The model's advantage grows with stress

CatBoost minus the rule-based score, same rule:

| Correlation | LGD | Average | Worst 5% of years |
|---|---|---|---|
| 0.00 | 30% | +0.67 | +0.71 |
| 0.15 | 30% | +0.68 | +2.06 |
| 0.30 | 30% | +0.68 | +2.82 |
| 0.30 | 50% | +1.00 | +3.83 |

The average column barely moves. The bad-year column climbs from 0.71 to 3.83, more than five times.

**What that means.** On a normal year the model earns you about three quarters of a point more. That is nice but not the argument. The argument is that when a lot of loans go bad at once, the portfolio the model picked holds up much better, and the worse the year, the bigger the difference.

The reason is that CatBoost tells individual loans apart. The rule-based scorer gives every loan in a bucket the same number, so it cannot avoid the specific loans inside a bucket that are the most likely to fail together. Those are exactly the loans that break a portfolio in a bad year.

### The harshest run

At double Basel's correlation with 50% LGD, the rule-based greedy portfolio's worst year returns **0.59%**. Half a point from losing money.

CatBoost's equivalent portfolio returns **6.12%** in its worst year. Ten times the cushion, on the same budget, in the same simulated year.

### No portfolio ever lost money

Zero losing years across all 36 portfolio-runs, including the harshest settings. A 7-year horizon on prime conforming loans is a hard thing to lose money on, and that is worth saying plainly rather than overstating the risk story.

### What correlation actually does

Look at the risk-sort rows to see it cleanly. At zero correlation the spread of outcomes is 0.02. At 0.30 it is 0.52, twenty-six times wider. The average return never moves.

**Correlation does not change what you earn. It changes how much the outcome can swing.** That is why independent draws were the wrong model in the first place: they produced a simulation where bad years essentially did not exist.

# Summary

**What this notebook did.** Took the six portfolios from the LP notebook and asked what happens to each one over the next seven years, 10,000 times.

---

### How the simulation works

Every loan gets a score each simulated year. If the score falls low enough, the loan defaults. The score is built from three pieces:

```
score = √0.1185 x national + √0.0315 x state + √0.85 x idiosyncratic
```

- **national**: one draw per year, hits every loan in the country
- **state**: one draw per state per year, hits only loans in that state
- **idiosyncratic**: one draw per loan, that borrower's own luck

The shared pieces are what make a bad year bad. When the national draw lands low, defaults cluster everywhere at once. That is what a recession looks like, and it is why we do not just flip an independent coin for each loan.

Total shared risk is 0.15, Basel's supervisory asset correlation for residential mortgages. We split it 79/21 between national and state. Every portfolio sees the same 10,000 years, so any difference between them comes from which loans they hold.

### What changed from the scaffold

The scaffold had only a national factor. Every loan got the same push regardless of where the house was, which made the state cap pointless by construction. There was no geographic risk to spread out.

We validated the new version on synthetic loans first. Four checks, all passed:

- Loans in the same state correlate more than loans in different states
- Turn the state factor off and that difference vanishes completely
- Set the national share to 100% and it reproduces the old model draw for draw
- The simulated default rate lands on the input probability

### The base run

Return on funded dollars over 7 years, 10,000 simulated outcomes.

| Score | Rule | Average | Bad year | Worst 5% | Worst |
|---|---|---|---|---|---|
| Rule-based | risk-sort | 24.22% | 23.25% | 22.64% | 19.36% |
| Rule-based | greedy-return | 29.43% | 25.78% | 24.18% | 16.37% |
| Rule-based | LP | 29.16% | 25.58% | 23.96% | 15.40% |
| CatBoost | risk-sort | 23.32% | 22.89% | 22.59% | 20.51% |
| CatBoost | greedy-return | 30.14% | 27.50% | 26.22% | 19.60% |
| CatBoost | LP | 29.83% | 27.24% | 25.98% | 18.82% |

"Bad year" is the return you beat 95% of the time. "Worst 5%" is the average across the 500 worst years.

**No portfolio lost money in any run**, here or anywhere else in the notebook.

---

## The three results

### 1. The concentration cap works, and still does not pay

The state shock is real. The gap between the greedy portfolio and the LP tracks the California factor at +0.78 correlation, and the national factor at +0.04. Greedy holds 28.2% of its budget in California, the LP holds 6%. The difference between those two portfolios is almost entirely a bet on one state.

The cap does buy protection. In the worst third of years, the LP's disadvantage shrinks from -0.29 to -0.24 once state risk is added. Its bad-year gap improves from -0.25 to -0.16. The two portfolios genuinely pull apart: their correlation drops from 0.99970 to 0.97127 and the spread between them widens six-fold.

**But the protection costs more than it returns.** The LP gives up upside in good years to get that cushion, and the trade comes out roughly even. It recovers about a third of its cost and never reaches break-even.

**The answer does not depend on the 79% figure.** We swept it from 100% national down to 50%. The average gap reads -0.31 at every single setting. Even at a split far beyond anything the literature supports, greedy still wins on every metric.

### 2. The model's value shows up in the bad years

CatBoost minus the rule-based score, same rule:

| Correlation | LGD | Average | Worst 5% |
|---|---|---|---|
| 0.00 | 30% | +0.67 | +0.71 |
| 0.15 | 30% | +0.68 | +2.06 |
| 0.30 | 50% | +1.00 | +3.83 |

On a normal year the model earns about three quarters of a point more. That is not the argument. The argument is the right-hand column: as conditions get worse, the gap grows more than five times.

**Why.** The rule-based scorer gives every loan in a bucket the same number, so it cannot avoid the specific loans inside a bucket most likely to fail together. Those are exactly the loans that break a portfolio in a bad year. CatBoost tells them apart.

At the harshest setting tested, the rule-based greedy portfolio's worst year returns 0.59%, half a point from losing money. CatBoost's returns 6.12%.

### 3. Risk-sorting is the safest and the worst

CatBoost risk-sort has the steadiest returns of all six portfolios by a wide margin, a spread of 0.22 against greedy's 1.31, and the best worst-year at 20.51%.

It also earns nearly 7 points less than every other rule.

Low default probability comes with low interest rates, because good credit gets good pricing. Sorting by risk funds the safest loans, which are also the lowest-yielding ones. A sharper risk model just does that more precisely.

---

### What the ranking survived

Twelve comparisons across the full sensitivity grid, asset correlation from 0.00 to 0.30 crossed with LGD at 30% and 50%. Greedy beat the LP in all twelve. CatBoost beat the rule-based score in all twelve.

### What we are not claiming

**States are drawn independently.** A bad year in California says nothing about Nevada. Real state housing markets move together, which would make spreading out worth less than modeled here, not more. So the finding holds under an assumption that already favors the cap.

**This is a 2017 book.** California was not in a downturn during that window. A regional collapse of the kind that hit Nevada, Arizona, and Florida in 2008 is outside anything this simulation can draw, because the state factor is a normal distribution and 2008 was not.

**The 79% split is a placeholder.** It comes from research on how much of state-level housing price movement is explained by a single national factor, pending the EDA track's check against FHFA data. The sweep is why that pending check does not hold up the conclusion.

### Files written

`prod_sim_returns.parquet` with 10,000 simulated returns for each of the six portfolios, and `prod_sim_summary.json` with the settings, the base run, the national-share sweep, the 10,000-year endpoint runs, and the full sensitivity grid.